# 🤖 Purchase Propensity Model

Feature engineering from `gold.user_metrics` + scikit-learn binary classifier.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, datediff, current_timestamp, lit

spark = (
    SparkSession.builder
    .appName('MLPropensity')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.hadoop.fs.s3a.endpoint', 'http://minio:9000')
    .config('spark.hadoop.fs.s3a.access.key', 'minioadmin')
    .config('spark.hadoop.fs.s3a.secret.key', 'minioadmin')
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.aws.credentials.provider',
            'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('Spark ready ✅')

## 1. Feature Engineering

In [ ]:
user_metrics = spark.read.format('delta').load('s3a://gold/user_metrics')
print(f'Users: {user_metrics.count()}')
user_metrics.printSchema()

In [ ]:
features_df = (
    user_metrics
    .withColumn('total_spend',           col('total_spend').cast('double'))
    .withColumn('avg_order_value',        col('avg_order_value').cast('double'))
    .withColumn('total_orders',           col('total_orders').cast('int'))
    .withColumn('completed_orders',       col('completed_orders').cast('int'))
    .withColumn('cancelled_orders',       col('cancelled_orders').cast('int'))
    .withColumn('unique_products_bought', col('unique_products_bought').cast('int'))
    .withColumn('total_clicks',           col('total_clicks').cast('int'))
    .withColumn('total_sessions',         col('total_sessions').cast('int'))
    # Target: high-value user (top 25% by spend)
    .withColumn('is_high_value', when(col('total_spend') > 200, 1).otherwise(0))
    .fillna(0)
)

feature_cols = [
    'total_orders', 'completed_orders', 'cancelled_orders',
    'avg_order_value', 'unique_products_bought',
    'total_clicks', 'total_sessions'
]

pandas_df = features_df.select(feature_cols + ['is_high_value', 'user_id']).toPandas()
print(f'Feature matrix shape: {pandas_df.shape}')
pandas_df.describe()

## 2. Train / Test Split

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, roc_auc_score, confusion_matrix,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

X = pandas_df[feature_cols].values
y = pandas_df['is_high_value'].values

print(f'Class balance — high-value: {y.mean():.1%}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

## 3. Model Training & Comparison

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
}

results = {}
for name, model in models.items():
    use_scaled = name == 'Logistic Regression'
    model.fit(X_train_s if use_scaled else X_train,
              y_train)
    proba = model.predict_proba(X_test_s if use_scaled else X_test)[:, 1]
    auc   = roc_auc_score(y_test, proba)
    results[name] = {'model': model, 'proba': proba, 'auc': auc, 'scaled': use_scaled}
    print(f'{name:25s} AUC = {auc:.4f}')

## 4. Best Model — Detailed Evaluation

In [ ]:
best_name = max(results, key=lambda k: results[k]['auc'])
best      = results[best_name]
print(f'Best model: {best_name}  (AUC = {best["auc"]:.4f})\n')

y_pred = best['model'].predict(
    X_test_s if best['scaled'] else X_test
)
print(classification_report(y_test, y_pred, target_names=['normal','high-value']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['normal','high-value']).plot(ax=axes[0])
axes[0].set_title(f'Confusion Matrix — {best_name}')

# Feature importance (if available)
if hasattr(best['model'], 'feature_importances_'):
    fi = pd.Series(best['model'].feature_importances_, index=feature_cols)
    fi.sort_values().plot(kind='barh', ax=axes[1])
    axes[1].set_title('Feature Importance')
else:
    coef = pd.Series(best['model'].coef_[0], index=feature_cols)
    coef.sort_values().plot(kind='barh', ax=axes[1])
    axes[1].set_title('Logistic Regression Coefficients')

plt.tight_layout()
plt.show()

## 5. Score All Users — Propensity Scores

In [ ]:
X_all   = pandas_df[feature_cols].values
use_s   = best['scaled']
scores  = best['model'].predict_proba(scaler.transform(X_all) if use_s else X_all)[:, 1]

scored = pandas_df[['user_id']].copy()
scored['propensity_score'] = scores
scored['segment'] = pd.cut(
    scores,
    bins=[0, 0.33, 0.66, 1.0],
    labels=['low','medium','high']
)

print('Top 10 highest propensity users:')
display(scored.sort_values('propensity_score', ascending=False).head(10))

print('\nSegment distribution:')
display(scored['segment'].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(scores, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
ax.set_xlabel('Propensity Score')
ax.set_ylabel('# Users')
ax.set_title('Purchase Propensity Score Distribution')
ax.axvline(0.33, color='orange', linestyle='--', label='Low/Medium')
ax.axvline(0.66, color='red',    linestyle='--', label='Medium/High')
ax.legend()
plt.tight_layout()
plt.show()